In [1]:
import pandas as pd
import torch
from tqdm import tqdm
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
from sklearn.metrics import f1_score, cohen_kappa_score, recall_score, accuracy_score, classification_report
from scipy.stats import spearmanr
from torch.utils.data import WeightedRandomSampler


# The following code will only execute
# successfully when compression is complete

import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohabenloughmari/miccai-task2-full")

print("Path to dataset files:", path)

import warnings
warnings.filterwarnings("ignore")

base_path = path
path = os.path.join(base_path, 'Task_2')

df_train = pd.read_csv(os.path.join(path, "df_task2_train.csv"))
df_val = pd.read_csv(os.path.join(path, "df_task2_val.csv"))
df_test = pd.read_csv(os.path.join(path, "df_task2_test.csv"))

device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')


100%|██████████| 6.72G/6.72G [00:45<00:00, 158MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mohabenloughmari/miccai-task2-full/versions/1


In [2]:

labels = df_train['label'].values.astype(int)
class_counts = np.bincount(labels)
sample_weights = 1.0 / class_counts[labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

n_labels = df_train['label'].nunique()

tabular_cols = ['case', 'label', 'LOCALIZER', 'split_type', 'image']
tab_train = df_train.drop(columns=tabular_cols)
tab_val = df_val.drop(columns=tabular_cols)
tab_test = df_test.drop(columns=tabular_cols)

cat_cols = tab_train.select_dtypes(include='object').columns.tolist()
num_cols = tab_train.select_dtypes(exclude='object').columns.tolist()

print(f"Categorical columns: {cat_cols}")
print(f"Numerical columns: {num_cols}")

tab_train_encoded = pd.get_dummies(tab_train, columns=cat_cols, dtype=np.float32)
tab_val_encoded = pd.get_dummies(tab_val, columns=cat_cols, dtype=np.float32)
tab_test_encoded = pd.get_dummies(tab_test, columns=cat_cols, dtype=np.float32)

tab_val_encoded = tab_val_encoded.reindex(columns=tab_train_encoded.columns, fill_value=0)
tab_test_encoded = tab_test_encoded.reindex(columns=tab_train_encoded.columns, fill_value=0)



Categorical columns: ['side_eye', 'sex']
Numerical columns: ['BScan', 'age', 'num_current_visit']


In [3]:

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [ ]:

class CustomImageDataset(Dataset):
    def __init__(self, dataframe, root_dir, tab_data, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.tab_data = torch.tensor(tab_data.values, dtype=torch.float32)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['image'])
        localiser_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['LOCALIZER'])

        image = Image.open(img_name).convert('RGB')
        localiser = Image.open(localiser_name).convert('RGB')

        if self.transform:
            image = self.transform(image)
            localiser = self.transform(localiser)

        label = self.dataframe.iloc[idx]['label']
        tab = self.tab_data[idx]
        return image, localiser, tab, label

In [5]:
train_ds = CustomImageDataset(df_train, os.path.join(path, 'train'), tab_train_encoded, transform=train_transform)  
val_ds = CustomImageDataset(df_val, os.path.join(path, 'val'), tab_val_encoded, transform=test_transform)  
test_ds = CustomImageDataset(df_test, os.path.join(path, 'test'), tab_test_encoded, transform=test_transform)  
  
labels = df_train['label'].values.astype(int)  
class_counts = np.bincount(labels)  
sample_weights = 1.0 / class_counts[labels]  
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))  
  
train_loader = DataLoader(train_ds, batch_size=64, sampler=sampler)  
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)  
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

In [6]:
!pip install open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 23.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.1 MB/s eta 0:00:00


In [7]:
import torch
import numpy as np
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from open_clip import create_model_from_pretrained, get_tokenizer


class FeatureExtractor:
    """Frozen BiomedCLIP feature extractor for OCT images."""
    def __init__(self, device):
        self.device = device
        self.model, self.preprocess = create_model_from_pretrained(
            'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
        )
        self.model = self.model.to(device).eval()
        for p in self.model.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def __call__(self, images):
        images = images.to(self.device)
        features = self.model.encode_image(images)
        return features.cpu().numpy()


def extract_features(dataloader, extractor):
    """Extract features from a dataloader using frozen BiomedCLIP."""
    all_img_feats = []
    all_labels = []

    for batch in tqdm(dataloader, desc="Extracting features"):
        images, localisers, tab, labels = batch

        img_feats = extractor(images)

        all_img_feats.append(img_feats)
        all_labels.append(labels.numpy() if isinstance(labels, torch.Tensor) else labels)

    img_feats = np.concatenate(all_img_feats)
    labels = np.concatenate(all_labels)

    return img_feats, labels


def build_pca_logreg_pipeline(train_loader, val_loader, device, n_components=3):
    """Full pipeline: BiomedCLIP extract → PCA(3) → LogisticRegression."""

    extractor = FeatureExtractor(device)

    print("Extracting train features...")
    X_train, y_train = extract_features(train_loader, extractor)
    print("Extracting val features...")
    X_val, y_val = extract_features(val_loader, extractor)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    pca = PCA(n_components=n_components)
    X_train_pca = pca.fit_transform(X_train)
    X_val_pca = pca.transform(X_val)
    print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")
    print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.4f}")

    class_weights_dict = {i: 1.0 / c for i, c in enumerate(class_counts)}
    clf = LogisticRegression(
        max_iter=1000,
        class_weight=class_weights_dict,
        multi_class='multinomial',
        solver='lbfgs',
    )
    clf.fit(X_train_pca, y_train)

    train_acc = clf.score(X_train_pca, y_train)
    val_acc = clf.score(X_val_pca, y_val)
    print(f"Train accuracy: {train_acc:.4f}")
    print(f"Val accuracy:   {val_acc:.4f}")

    return scaler, pca, clf


# --- Usage ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler, pca, clf = build_pca_logreg_pipeline(train_loader, val_loader, device, n_components=10)

open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

Extracting train features...


Extracting features: 100%|██████████| 127/127 [08:33<00:00,  4.04s/it]


Extracting val features...


Extracting features: 100%|██████████| 60/60 [03:47<00:00,  3.79s/it]


PCA explained variance ratio: [0.1656212  0.12420556 0.070633   0.05284918 0.0497083  0.03678725
 0.03129805 0.02798922 0.02606517 0.02303784]
Total variance explained: 0.6082
Train accuracy: 0.4228
Val accuracy:   0.1727


In [8]:
from torchvision import models

class ImageEncoder(nn.Module):
    def __init__(self, embed_dim=768):
        super().__init__()
        weights = models.EfficientNet_V2_S_Weights.DEFAULT
        self.backbone = models.efficientnet_v2_s(weights=weights)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        self.fc = nn.Linear(in_features, embed_dim)
        for p in self.backbone.parameters():
            p.requires_grad = True

    def forward(self, x):
        features = self.backbone(x)
        return self.fc(features)


class LocaliserEncoder(nn.Module):
    def __init__(self, embed_dim=768):
        super().__init__()
        weights = models.EfficientNet_V2_S_Weights.DEFAULT
        self.backbone = models.efficientnet_v2_s(weights=weights)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        self.fc = nn.Linear(in_features, embed_dim)
        for p in self.backbone.parameters():
            p.requires_grad = True

    def forward(self, x):
        features = self.backbone(x)
        return self.fc(features)


class TabularMLP(nn.Module):
    def __init__(self, tab_dim, d_model, dropout=0.3):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(tab_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, d_model),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, tab):
        return self.mlp(tab)  # (B, d_model)


class OCT_Pred(nn.Module):
    def __init__(self, n_labels, tab_dim, embed_dim=768, d_model=None, dropout=0.3):
        super().__init__()
        if d_model is None:
            d_model = embed_dim

        self.image_encoder = ImageEncoder(embed_dim)
        self.localiser_encoder = LocaliserEncoder(embed_dim)
        self.tab_encoder = TabularMLP(tab_dim, d_model, dropout)

        fusion_dim = 2 * embed_dim + d_model

        self.mlp = nn.Sequential(
            nn.Linear(fusion_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, n_labels),
        )

    def forward(self, image, localiser, tab):
        img_features = self.image_encoder(image)
        loc_features = self.localiser_encoder(localiser)
        tab_features = self.tab_encoder(tab)

        combined = torch.cat([img_features, loc_features, tab_features], dim=-1)
        output = self.mlp(combined)
        return output, combined


class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)

In [9]:
from sklearn.metrics import classification_report
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_predicted = []

    pbar = tqdm(loader, desc="Training", leave=False)
    for images, localisers, tab, labels in pbar:
        images = images.to(device)
        localisers = localisers.to(device)
        tab = tab.to(device)
        labels = labels.long().to(device)

        optimizer.zero_grad()
        outputs, _ = model(images, localisers, tab)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

        all_labels.extend(labels.cpu().numpy())
        all_predicted.extend(predicted.cpu().numpy())

    accuracy = 100. * correct / total
    print(classification_report(all_labels, all_predicted))
    return running_loss / len(loader), accuracy

In [10]:
from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix, cohen_kappa_score, classification_report
import numpy as np

def specificity(class_ground_truth, class_prediction):
    eps = 0.000001
    cnf_matrix = confusion_matrix(class_ground_truth, class_prediction)
    FP = cnf_matrix.sum(axis=0) - np.diag(cnf_matrix)
    FN = cnf_matrix.sum(axis=1) - np.diag(cnf_matrix)
    TP = np.diag(cnf_matrix)
    TN = cnf_matrix.sum() - (FP + FN + TP)
    FP, FN, TP, TN = FP.astype(float), FN.astype(float), TP.astype(float), TN.astype(float)
    spe = TN / (TN + FP + eps)
    return spe.mean()

def evaluate(model, dataloader, criterion, device, show_report=True):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Evaluation', leave=False)
        for images, localisers, tab, labels in pbar:
            images = images.to(device)
            localisers = localisers.to(device)
            tab = tab.to(device)
            labels = labels.long().to(device)

            outputs, _ = model(images, localisers, tab)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = 100. * correct / total
    f1 = f1_score(all_labels, all_preds, average='micro')
    rk_corr = matthews_corrcoef(all_labels, all_preds)
    spec = specificity(all_labels, all_preds)
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    score = 0.1 * f1 + 0.1 * spec + 0.6 * qwk + 0.2 * rk_corr

    if show_report:
        print("\n" + "="*80)
        print("CLASSIFICATION REPORT - Per-Class Performance:")
        print("="*80)
        print(classification_report(all_labels, all_preds,
                                    target_names=['Class 0', 'Class 1', 'Class 2'],
                                    digits=4))
        print("="*80 + "\n")

    return {
        'loss': total_loss / len(dataloader),
        'accuracy': accuracy,
        'F1_score': f1,
        'Rk-correlation': rk_corr,
        'Specificity': spec,
        'Quadratic-weighted_Kappa': qwk,
        'score': score
    }




In [11]:
#class FocalLoss(nn.Module):
#import torch.nn.functional as F
#    def __init__(self, gamma=2):
#        super().__init__()
#        self.gamma = gamma
#        
#    def forward(self, inputs, targets):
#        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
#        pt = torch.exp(-ce_loss)
#        focal_loss = (1 - pt) ** self.gamma * ce_loss
#        return focal_loss.mean()
#criterion = FocalLoss(gamma=2)


In [12]:
model = OCT_Pred(n_labels=n_labels, tab_dim=7).to(device)
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
num_epochs = 5

best_score = -float('inf')

for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"{'='*50}")

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.2f}%")
    print(f"Val Score: {val_metrics['score']:.4f}")

    if val_metrics['score'] > best_score:
        best_score = val_metrics['score']
        best_model=model
        print(f"New best model saved! Score: {best_score:.4f}")


Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 199MB/s] 



Epoch 1/5


OutOfMemoryError: CUDA out of memory. Tried to allocate 46.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 3.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 14.06 GiB is allocated by PyTorch, and 378.22 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
results = evaluate(best_model, test_loader, criterion, device)
print(f"Accuracy: {results['accuracy']:.2f}%")
print(f"F1 Score: {results['F1_score']:.4f}")
print(f"Quadratic Weighted Kappa: {results['Quadratic-weighted_Kappa']:.4f}")
print(f"Rk Correlation (Spearman): {results['Rk-correlation']:.4f}")
print(f"Specificity: {results['Specificity']:.4f}")
print(f"Score: {results['score']:.4f}")


CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     1.0000    0.0329    0.0637       578
     Class 1     0.8448    0.9960    0.9142      4810
     Class 2     0.0000    0.0000    0.0000       321

    accuracy                         0.8425      5709
   macro avg     0.6149    0.3430    0.3260      5709
weighted avg     0.8130    0.8425    0.7767      5709


Accuracy: 84.25%
F1 Score: 0.8425
Quadratic Weighted Kappa: 0.0406
Rk Correlation (Spearman): 0.0840
Specificity: 0.6725
Score: 0.1926


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
#from google.colab import files
#files.download('model.pth')
import torch
torch.save(best_model.state_dict(), '/content/drive/MyDrive/best_model_mlp.pth')

Mounted at /content/drive
